# MoLoRAG Baseline Verification (Google Colab)
This notebook runs the first baseline (M3DocRAG Base Retriever) on the `MMLong` dataset.
**Instructions:**
1. Go to **Runtime -> Change runtime type** and ensure **T4 GPU** is selected.
2. Run all cells sequentially.

In [ ]:
# 1. Clone the repository and install dependencies
!apt-get update && apt-get install -y poppler-utils
!git clone https://github.com/WxxShirley/MoLoRAG.git
%cd MoLoRAG
%pip install "transformers>=4.49.0" torchvision==0.20.1 qwen-vl-utils==0.0.8 accelerate langchain==0.3.19 langchain-community==0.3.18 langchain-core==0.3.37 langchain-text-splitters==0.3.6 PyMuPDF==1.25.3 pypdf==5.3.0 pdf2image==1.17.0
%pip install colpali_engine==0.3.8 colbert-ai==0.2.21 --no-deps


In [ ]:
# 2. Download a subset of the dataset (we will use MMLong for speed)
from huggingface_hub import snapshot_download
snapshot_download(repo_id='xxwu/MoLoRAG', repo_type='dataset', allow_patterns='MMLong/*', local_dir='./dataset/')


In [ ]:
# 3. Run Step 1: Indexing for Base Retriever (M3DocRAG)
import os
os.chdir('/content/MoLoRAG/VLMRetriever')
!python3 index.py --dataset MMLong --save_img


In [ ]:
# 4. Run Step 2: Retrieve relevant pages
!python3 retrieve.py --dataset MMLong --method base


In [ ]:
# 5. Run Step 3: LVLM direct inference Baseline on the retrieved documents using QwenVL (Base T4 usage)
os.chdir('/content/MoLoRAG')
!TOKENIZERS_PARALLELISM=false python3 -u main.py --dataset=MMLong --model_name=QwenVL-7B --device=cuda:0 --retriever=base --topk=3


In [ ]:
# 6. Evaluate the results of the baseline
os.chdir('/content/MoLoRAG/evaluate')
!python3 main_eval.py --dataset MMLong --model_name QwenVL-7B --retriever base
!cat ../results/MMLong/metrics/QwenVL-7B_base_eval.json
